# Tutorial 03: Solving a PDE

In Tutorial 02 we used a PINN to enforce an ODE ($y' = \cos x$) and gained extrapolation on a 1D problem. Now we tackle a **partial differential equation** — the 1D diffusion equation — which introduces a second spatial variable ($t$) and second-order derivatives.

#### The problem

$$\frac{\partial y}{\partial t} = \frac{\partial^2 y}{\partial x^2} - e^{-t}\!\left(\sin(\pi x) - \pi^2 \sin(\pi x)\right), \quad x \in [-1, 1],\; t \in [0, 1]$$

with **initial condition** (IC) and **boundary conditions** (BC):

$$y(x, 0) = \sin(\pi x), \quad y(-1, t) = 0, \quad y(1, t) = 0$$

The exact solution is $y(x, t) = e^{-t} \sin(\pi x)$, so we can measure how well the PINN does.

#### What's new compared to Tutorial 02

| | Tutorial 02 (ODE) | Tutorial 03 (PDE) |
|---|---|---|
| **Input** | $x$ (1D) | $(x, t)$ (2D) |
| **Equation** | $y' = \cos x$ | $y_t = y_{xx} + $ source term |
| **Derivatives** | 1st order | 1st *and* 2nd order |
| **Training data** | labeled $(x, y)$ pairs | IC + BC points only |

#### PINN loss structure

The total loss has two parts:

- **Data loss** $\mathcal{L}_{\text{data}}$: MSE on IC + BC points (where we know the exact values)
- **Physics loss** $\mathcal{L}_{\text{PDE}}$: PDE residual at random collocation points in the interior

$$\mathcal{L}_{\text{PDE}} = \frac{1}{M}\sum_{j=1}^{M}\left| y_t - y_{xx} + e^{-t}(\sin \pi x - \pi^2 \sin \pi x) \right|^2_{(x_j, t_j)}$$

The trainer handles $\mathcal{L}_{\text{data}}$; the model injects $\mathcal{L}_{\text{PDE}}$ via `model_loss`.

---

### Table of Contents

- [Step 0: Setups](#step0)
- [Step 1: Prepare data](#step1)
- [Step 2: Define the PINN model](#step2)
- [Step 3: Train and evaluate](#step3)
- [Takeaway](#takeaway)
- [References](#references)

---

<a id='step0'></a>
### Step 0: Setups
---

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from utils import add_necessary_paths, check_torch
add_necessary_paths()

import utils.u03 as u
import talos as ta
import numpy as np

check_torch()
ta.set_seed()

<a id='step1'></a>
### Step 1: Prepare data
---

Unlike Tutorial 01 where we had labeled $(x, y)$ pairs everywhere, here the only labeled data comes from the **boundaries** of the domain:

- **IC** (bottom edge): $y(x, 0) = \sin(\pi x)$ — 200 points along $t = 0$
- **BC** (left/right edges): $y(-1, t) = y(1, t) = 0$ — 100 points each

The interior of the domain has **no labeled data** — the PINN must learn it purely from the PDE.

In [ ]:
# (1) Generate IC + BC training points.
X_train, Y_train = u.generate_icbc_data()
train_set = ta.Dataset(X_train, Y_train, name='IC+BC')
train_set.report()

# (2) Generate a dense test grid with exact solution.
X_test, Y_test, x_grid, t_grid = u.generate_test_grid()
test_set = ta.Dataset(X_test, Y_test, name='Full Grid')
test_set.report()

In [ ]:
# Visualize IC/BC training points on the domain
u.plot_icbc_points(X_train)

In [ ]:
# Visualize the exact solution
u.plot_contour(x_grid, t_grid, Y_test.flatten(), title='Exact Solution: $e^{-t} \\sin(\\pi x)$')

<a id='step2'></a>
### Step 2: Define the PINN model
---

We subclass `MLP` and override `model_loss` to compute the PDE residual. The key steps:

1. **Sample collocation points** $(x, t)$ randomly in the domain interior
2. **Forward pass** to get $\hat{y}(x, t)$
3. **Compute derivatives** $\hat{y}_t$ and $\hat{y}_{xx}$ via `torch.autograd.grad`
4. **Return the PDE residual** $\text{mean}|\hat{y}_t - \hat{y}_{xx} + \text{source}|^2$

Note that computing $\hat{y}_{xx}$ requires differentiating **twice** — we first get $\hat{y}_x$, then differentiate again. Both calls use `create_graph=True` so the result stays in the computation graph for backpropagation.

In [ ]:
import torch

class DiffusionPINN(ta.model.torch_zoo.MLP):
  """MLP with physics loss for the 1D diffusion equation."""

  def __init__(self, *args, n_collocation=10000,
               x_min=-1, x_max=1, t_min=0, t_max=1, **kwargs):
    super().__init__(*args, **kwargs)
    self.n_collocation = n_collocation
    self.x_min, self.x_max = x_min, x_max
    self.t_min, self.t_max = t_min, t_max

  def model_loss(self, X, outputs, Y):
    """PDE residual: y_t - y_xx + source = 0."""
    # (1) Sample random collocation points in the domain.
    x_c = torch.rand(self.n_collocation, 1, device=X.device) * (self.x_max - self.x_min) + self.x_min
    t_c = torch.rand(self.n_collocation, 1, device=X.device) * (self.t_max - self.t_min) + self.t_min
    x_c.requires_grad_(True)
    t_c.requires_grad_(True)
    xt_c = torch.cat([x_c, t_c], dim=1)

    # (2) Forward pass at collocation points.
    y_c = self.forward(xt_c)

    # (3) First derivatives: dy/dx and dy/dt.
    ones = torch.ones_like(y_c)
    dy_dx = torch.autograd.grad(y_c, x_c, ones, create_graph=True)[0]
    dy_dt = torch.autograd.grad(y_c, t_c, ones, create_graph=True)[0]

    # (4) Second derivative: d²y/dx².
    dy_dxx = torch.autograd.grad(dy_dx, x_c, ones, create_graph=True)[0]

    # (5) Source term: e^{-t} (sin(pi*x) - pi^2 sin(pi*x)).
    source = torch.exp(-t_c) * (torch.sin(np.pi * x_c) - np.pi**2 * torch.sin(np.pi * x_c))

    # (6) PDE residual: y_t - y_xx + source = 0.
    residual = dy_dt - dy_dxx + source
    return torch.mean(residual ** 2)

<a id='step3'></a>
### Step 3: Train and evaluate
---

We use a small network (`[2, 32, 32, 1]`) with Adam optimizer. The data loss (MSE) fits the IC/BC points; the `model_loss` enforces the PDE everywhere else.

In [ ]:
# Initialize model
pinn = DiffusionPINN(2, [32, 32], 1, activation='tanh')
print(pinn)

# Train with Adam optimizer
trainer = u.get_trainer(pinn, loss_fn='mse', optimizer='adam', lr=1e-3,
                        print_every=2000)
trainer.train(train_set, 20000)

In [ ]:
# Predict on the full test grid
Y_pred = pinn.predict(test_set)

# Side-by-side comparison: exact, predicted, and error
u.plot_comparison(x_grid, t_grid, Y_test, Y_pred)

<a id='takeaway'></a>
### Takeaway

With the same `model_loss` pattern from Tutorial 02, we solved a PDE — the 1D diffusion equation — using only IC and BC data. The interior solution was learned entirely from the physics constraint.

Key new elements in this tutorial:

- **2D input** $(x, t)$: the network maps two variables to one output
- **Second-order derivatives**: $y_{xx}$ requires two successive `autograd.grad` calls
- **No interior labels**: the training set consists only of boundary data; the PDE residual drives learning in the interior

The Talos workflow stayed the same: subclass an existing model, override `model_loss`, and train with the standard trainer.

<a id='references'></a>
### References

This tutorial was adapted from the diffusion equation example in:

**NABLA-SciML: Learning Physics-Informed Machine Learning in Python**  
Juan Diego Toscano et al.  
[4_DiffusionEquation.ipynb](https://github.com/jdtoscano94/NABLA-SciML/blob/main/Learning-PIML-in-Python-PINNs-DeepONets-RBA/Tutorials/PINNs/4_DiffusionEquation.ipynb)